In [1]:
import os
import pandas as pd

folder_path = r"C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\Bluestock-project-repo-1\bluestock_mf_capstone\data\processed"

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        cols = pd.read_csv(
            os.path.join(folder_path, file),
            nrows=0
        ).columns.tolist()

        print(f"\n{file}")
        print(cols)


alpha_beta_data.csv
['amfi_code', 'alpha_nifty50', 'beta_nifty50', 'alpha_nifty100', 'beta_nifty100']

aum_by_fund_house_cleaned.csv
['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']

benchmark_indices_cleaned.csv
['date', 'index_name', 'close_value']

category_inflows_cleaned.csv
['month', 'category', 'net_inflow_crore']

fund_master_cleaned.csv
['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']

fund_scorecard.csv
['amfi_code', 'cagr_3y', 'sharpe_ratio', 'alpha', 'expense_ratio', 'max_drawdown', 'cagr_rank', 'sharpe_rank', 'alpha_rank', 'expense_rank', 'dd_rank', 'fund_score']

industry_folio_count_cleaned.csv
['month', 'total_folios_crore', 'equity_folios_crore', 'debt_folios_crore', 'hybrid_folios_crore', 'others_folios_crore']

investor_transactions_cleaned.csv
['investor_id'

In [2]:
nav_df = pd.read_csv(r"C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\Bluestock-project-repo-1\bluestock_mf_capstone\data\processed\nav_history_cleaned.csv")

In [ ]:
import pandas as pd


nav_df['date'] = pd.to_datetime(nav_df['date'])

nav_df = nav_df.sort_values(
    ['amfi_code', 'date']
)

# Daily Returns

nav_df['daily_return'] = (
    nav_df.groupby('amfi_code')['nav']
    .pct_change()
)

# VaR (95%)

var_df = (
    nav_df.groupby('amfi_code')['daily_return']
    .quantile(0.05)
    .reset_index(name='var_95')
)

# CVaR (95%)

def calc_cvar(x):

    var = x.quantile(0.05)

    return x[x <= var].mean()

cvar_df = (
    nav_df.groupby('amfi_code')['daily_return']
    .apply(calc_cvar)
    .reset_index(name='cvar_95')
)

# Final Report

var_cvar_report = var_df.merge(
    cvar_df,
    on='amfi_code',
    how='inner'
)

var_cvar_report.to_csv(
    'C:\\Users\\Asus\\OneDrive\\Desktop\\Bluestock_intern\\Bluestock-project-repo-1\\bluestock_mf_capstone\\Bluestock-project-repo-1\\bluestock_mf_capstone\\data\\processed\\var_cvar_report.csv',
    index=False
)

print(var_cvar_report.head())

   amfi_code    var_95   cvar_95
0     100016 -0.012884 -0.016768
1     100025 -0.003338 -0.004581
2     100033 -0.016902 -0.021850
3     101206 -0.012173 -0.016075
4     101207 -0.023915 -0.030289


In [7]:
fund_scorecard = pd.read_csv(r"C:\Users\Asus\OneDrive\Desktop\Bluestock_intern\Bluestock-project-repo-1\bluestock_mf_capstone\Bluestock-project-repo-1\bluestock_mf_capstone\data\processed\fund_scorecard.csv")

In [8]:
import matplotlib.pyplot as plt
import numpy as np


nav_df['rolling_sharpe'] = (
    nav_df.groupby('amfi_code')['daily_return']
    .transform(
        lambda x:
        (
            x.rolling(90).mean()
            /
            x.rolling(90).std()
        )
        * np.sqrt(252)
    )
)

# Top 5 Funds

top5 = (
    fund_scorecard
    .sort_values(
        'fund_score',
        ascending=False
    )
    .head(5)['amfi_code']
)

plt.figure(figsize=(15,8))

for fund in top5:

    temp = nav_df[
        nav_df['amfi_code'] == fund
    ]

    plt.plot(
        temp['date'],
        temp['rolling_sharpe'],
        label=str(fund)
    )

plt.title('Rolling 90-Day Sharpe Ratio')
plt.xlabel('Date')
plt.ylabel('Sharpe Ratio')
plt.legend()

plt.savefig(
    'C:\\Users\\Asus\\OneDrive\\Desktop\\Bluestock_intern\\Bluestock-project-repo-1\\bluestock_mf_capstone\\Bluestock-project-repo-1\\bluestock_mf_capstone\\reports\\rolling_sharpe_chart.png',
    dpi=300,
    bbox_inches='tight'
)

plt.close()

In [9]:
transactions_df = pd.read_csv("C:\\Users\\Asus\\OneDrive\\Desktop\\Bluestock_intern\\Bluestock-project-repo-1\\bluestock_mf_capstone\\Bluestock-project-repo-1\\bluestock_mf_capstone\\data\\processed\\investor_transactions_cleaned.csv")

In [10]:
transactions_df['transaction_date'] = pd.to_datetime(
    transactions_df['transaction_date']
)

# First Transaction

transactions_df['cohort_year'] = (
    transactions_df.groupby('investor_id')
    ['transaction_date']
    .transform('min')
    .dt.year
)

# Avg Investment + Total Investment

cohort_summary = (
    transactions_df.groupby('cohort_year')
    .agg(
        avg_investment=('amount_inr','mean'),
        total_invested=('amount_inr','sum')
    )
    .reset_index()
)

# Most Popular Fund

top_fund = (
    transactions_df.groupby(
        ['cohort_year','amfi_code']
    )
    .size()
    .reset_index(name='count')
)

top_fund = (
    top_fund.sort_values('count')
    .groupby('cohort_year')
    .tail(1)
)

cohort_analysis = cohort_summary.merge(
    top_fund[['cohort_year','amfi_code']],
    on='cohort_year',
    how='left'
)

cohort_analysis.rename(
    columns={
        'amfi_code':'top_fund'
    },
    inplace=True
)

print(cohort_analysis)

   cohort_year  avg_investment  total_invested  top_fund
0         2024   107422.541832      3491125187    148568
1         2025   109158.577061        30455243    119599


In [11]:
sip_df = transactions_df[
    transactions_df['transaction_type']
    == 'SIP'
].copy()

# Investors with >= 6 SIPs

sip_df = (
    sip_df.groupby('investor_id')
    .filter(lambda x: len(x) >= 6)
)

sip_df = sip_df.sort_values(
    ['investor_id','transaction_date']
)

# Gap

sip_df['gap_days'] = (
    sip_df.groupby('investor_id')
    ['transaction_date']
    .diff()
    .dt.days
)

sip_continuity = (
    sip_df.groupby('investor_id')
    ['gap_days']
    .mean()
    .reset_index()
)

sip_continuity.rename(
    columns={
        'gap_days':'avg_gap_days'
    },
    inplace=True
)

sip_continuity['status'] = np.where(
    sip_continuity['avg_gap_days'] > 35,
    'at-risk',
    'healthy'
)

print(sip_continuity.head())

  investor_id  avg_gap_days   status
0   INV000004     85.400000  at-risk
1   INV000008     70.400000  at-risk
2   INV000010     64.800000  at-risk
3   INV000011     40.166667  at-risk
4   INV000012     57.000000  at-risk


In [14]:
holdings_df = pd.read_csv(
    "C:\\Users\\Asus\\OneDrive\\Desktop\\Bluestock_intern\\Bluestock-project-repo-1\\bluestock_mf_capstone\\Bluestock-project-repo-1\\bluestock_mf_capstone\\data\\processed\\portfolio_holdings_cleaned.csv"
)

In [15]:
sector_weights = (
    holdings_df
    .groupby(
        ['amfi_code','sector']
    )['weight_pct']
    .sum()
    .reset_index()
)

sector_hhi = (
    sector_weights
    .groupby('amfi_code')['weight_pct']
    .apply(
        lambda x:
        np.sum(
            (x/100)**2
        )
    )
    .reset_index(name='hhi')
)

sector_hhi = sector_hhi.sort_values(
    'hhi',
    ascending=False
)

print(sector_hhi.head())

    amfi_code       hhi
11     119092  0.296769
30     148569  0.254992
27     125498  0.253155
6      102887  0.251383
32     149323  0.241077


In [16]:
highest_var = (
    var_cvar_report
    .sort_values('var_95')
    .head(1)
)

highest_cvar = (
    var_cvar_report
    .sort_values('cvar_95')
    .head(1)
)

largest_cohort = (
    cohort_analysis
    .sort_values(
        'total_invested',
        ascending=False
    )
    .head(1)
)

sip_continuity_rate = (
    (
        sip_continuity['status']
        == 'healthy'
    ).mean()
    * 100
)

most_concentrated = (
    sector_hhi
    .sort_values(
        'hhi',
        ascending=False
    )
    .head(1)
)

print(
    f"Highest VaR Fund: {highest_var['amfi_code'].iloc[0]}"
)

print(
    f"Highest CVaR Fund: {highest_cvar['amfi_code'].iloc[0]}"
)

print(
    f"Largest Investor Cohort: {largest_cohort['cohort_year'].iloc[0]}"
)

print(
    f"SIP Continuity Rate: {sip_continuity_rate:.2f}%"
)

print(
    f"Most Concentrated Fund: {most_concentrated['amfi_code'].iloc[0]}"
)

Highest VaR Fund: 101207
Highest CVaR Fund: 101207
Largest Investor Cohort: 2024
SIP Continuity Rate: 2.20%
Most Concentrated Fund: 119092


### Highest VaR Fund: 101207
### Highest CVaR Fund: 101207
### Largest Investor Cohort: 2024
### SIP Continuity Rate: 2.20%
### Most Concentrated Fund: 119092